In [1]:
import chromadb
import json
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com"
)

In [2]:
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")

all_chunks = []
doc_ids = [
    "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c",
    "5320ce59-13a9-4459-b00f-274e45684042",
    "054df5ac-44ee-49ff-b4a3-fd78a48c7937",
]

for doc_id in doc_ids:
    collection = chroma_client.get_collection(f"doc_{doc_id}")
    results = collection.get()

    for chunk_id, metadata in zip(results["ids"], results["metadatas"]):
        all_chunks.append(
            {"chunk_id": chunk_id, "doc_id": doc_id, "text": metadata["text"][:500]}
        )

In [3]:
prompt = f"""Generate 15 questions based on these chunks.

For each chunk, create a question that can ONLY be answered from that exact chunk.

Format as JSON array: 
[
 {{
  "query": "question based on chunk",
  "expected_answer": "short answer from chunk",
  "doc_id": "chunk's doc_id",
  "relevant_chunk_id": "chunk's chunk_id",
  "relevant_chunk": true
 }}
] 

Also add 5 unanswerable questions with relevant_chunk: false and relevant_chunk_id: null 

Chunks: {json.dumps(all_chunks, indent=2)} 

Return ONLY valid JSON. No explanation."""

In [4]:
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
)

In [5]:
op=response.choices[0].message.content
op=op.strip("```json")
op=op.removeprefix("\n")
op=op.removesuffix("\n")
op

'[\n {\n  "query": "What was the index case location of the COVID-19 pandemic?",\n  "expected_answer": "Wuhan, China",\n  "doc_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c",\n  "relevant_chunk_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c_chunk_0",\n  "relevant_chunk": true\n },\n {\n  "query": "On what date did the WHO describe the COVID-19 outbreak as a pandemic?",\n  "expected_answer": "11 March 2020",\n  "doc_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c",\n  "relevant_chunk_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c_chunk_1",\n  "relevant_chunk": true\n },\n {\n  "query": "When did the WHO declare that the public health emergency caused by COVID-19 had ended?",\n  "expected_answer": "May 2023",\n  "doc_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c",\n  "relevant_chunk_id": "ab4adbd5-7eee-4e11-bed2-7f8688dddd4c_chunk_2",\n  "relevant_chunk": true\n },\n {\n  "query": "When were COVID-19 vaccines first deployed to the general public?",\n  "expected_answer": "December 2020",\n  "doc_id": "ab4ad

In [6]:
test_queries = json.loads(op)

with open("test_queries.json", "w") as f:
    json.dump(test_queries, f, indent=2)